# Fused Cross-Entropy Benchmark: Triton vs Hand-CUDA vs PyTorch

Four implementations, profiled with Nsight Systems + Nsight Compute on T4 and Blackwell (B200). This notebook is the single orchestration point: env check, compile, parity, timing sweep, ncu sweep, nsys sweep, merge, plot.

**Run order**: top-to-bottom. Every cell is idempotent; re-running after a session drop picks up where it left off.

## Runbook cheat sheet (B200 5-hour burst)

| Window | Task | Cell section |
|---|---|---|
| 00:00 -- 00:30 | Spin instance, SSH in, `git clone`, start Jupyter, tunnel port 8888. Run Sections 1--3 (env, grid, compile+smoke). Kill at 00:40 if smoke fails. | 1, 2, 3 |
| 00:30 -- 00:45 | Run the ncu permission-unlock cell. Run one `run_ncu(...)` call to verify. If denied, set `SKIP_NCU = True` and fall through. | 4, 5 (first capture only) |
| 00:45 -- 02:30 | Run the full ncu sweep (12 shapes x 4 impls = 48 captures). Do NOT touch the notebook. | 5 |
| 02:30 -- 03:00 | Run the nsys sweep + CUDA-Graph variant cell. | 6, 6b |
| 03:00 -- 03:45 | Exactly ONE bonus experiment (autotune amortization OR NVML power). | 7 |
| 03:45 -- 04:15 | Re-run the 4 canonical shapes with `force=True` for hero-table stability. | 5 (force rerun) |
| 04:15 -- 04:45 | Run merge + plot cells. Export tarball. `git push` from the notebook. | 8, 9, 10 |
| 04:45 -- 05:00 | Terminate the instance from the provider console. Verify billing stopped. | out-of-band |

## 0. Bootstrap (Colab-friendly)

If running on Google Colab, this cell clones the repo and installs requirements. On a self-managed box where the repo is already cloned and the venv is active, this is a no-op.

In [ ]:
import os, pathlib, sys

_IN_COLAB = 'google.colab' in sys.modules
_REPO_URL = os.environ.get('FUSED_XENT_REPO', 'https://github.com/sriharshapy/Fused-Cross-Entropy.git')

if _IN_COLAB:
    if not pathlib.Path('/content/Fused-CrossEntropy-Benchmark').exists():
        !git clone {_REPO_URL} /content/Fused-CrossEntropy-Benchmark
    %cd /content/Fused-CrossEntropy-Benchmark
    !pip -q install -r requirements.txt

# Locate repo root = parent of 'bench/' dir. Works in Colab, VS Code, JupyterLab.
_cwd = pathlib.Path.cwd()
REPO_ROOT = _cwd if (_cwd / 'kernels').exists() else _cwd.parent
assert (REPO_ROOT / 'kernels').exists(), f'kernels/ not found relative to {REPO_ROOT}'
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT =', REPO_ROOT)

## 1. Environment check

Confirms GPU, CUDA toolkit, profiler binaries, and PyTorch / Triton versions. If any of these fail on the B200 box, STOP and fix before spending sweep time.

Also sets the global `ARCH` (`'t4'` or `'b200'`) that every downstream cell keys off.

In [ ]:
!nvidia-smi | head -n 20

## 1b. Install nsys / ncu if missing (Colab + cloud VMs)

On Colab the base image ships with `ncu` but *not* `nsys`; on bare cloud VMs sometimes neither is present. This cell mirrors the pattern from [Sigmoid-TopK-Fusion](https://github.com/sriharshapy/Sigmoid-TopK-Fusion/blob/main/sigmoid_topk_benchmark_ncu.ipynb) — download the NVIDIA DevTools `.deb` and `dpkg -i` it. Safe to re-run: if the tool already resolves on `PATH` it's a no-op.

If you're on a Windows dev box or a provider that supplies nsys via a module system (`module load nsight-systems`), skip this cell.

In [ ]:
import shutil, platform, os

# Pinned .deb URL copied from Sigmoid-TopK-Fusion's benchmark notebook. If a
# newer Nsight Systems wheel lands and you've tested it, bump the URL here.
NSYS_DEB_URL = 'https://developer.download.nvidia.com/devtools/repos/ubuntu2204/amd64/NsightSystems-linux-cli-public-2024.6.1.90-3490548.deb'


def install_nsys():
    if shutil.which('nsys'):
        print('nsys: present at', shutil.which('nsys'))
        return
    if platform.system() != 'Linux':
        print('nsys missing; auto-install only on Linux. Manual download:', NSYS_DEB_URL)
        return
    print('nsys missing; installing from', NSYS_DEB_URL)
    # Same three-command sequence as Sigmoid-TopK-Fusion's cell:
    #   wget -> dpkg -i  (||  apt -f install to fix deps)
    os.system(f'wget -q {NSYS_DEB_URL} -O /tmp/nsys.deb')
    os.system('sudo -n dpkg -i /tmp/nsys.deb 2>/dev/null || dpkg -i /tmp/nsys.deb')
    os.system('sudo -n apt-get -f install -y 2>/dev/null || apt-get -f install -y')
    print('nsys --version:')
    os.system('nsys --version')


def check_ncu():
    # ncu ships with the CUDA toolkit; on Colab/most cloud images it's already\n    # present. If missing, document the fix rather than silently auto-installing,
    # because the ncu package tends to pull a lot of CUDA runtime deps.
    if shutil.which('ncu'):
        print('ncu: present at', shutil.which('ncu'))
    else:
        print('ncu NOT found on PATH. If you are on Ubuntu, install with:')
        print('  sudo apt-get install -y nsight-compute-2024.3.2')
        print('  (or `apt-cache search nsight-compute` to find the latest).')
        print('If you cannot install ncu, set `SKIP_NCU = True` below and continue.')


install_nsys()
check_ncu()

In [ ]:
import shutil, subprocess

def _which(name):
    p = shutil.which(name)
    return p or '(missing)'

def _ver(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True).strip().splitlines()[0]
    except Exception as e:
        return f'ERROR: {e}'

print('nvcc          :', _which('nvcc'))
print('ncu           :', _which('ncu'))
print('nsys          :', _which('nsys'))
print()
print('nvcc --version:', _ver('nvcc --version | tail -n 1'))
print('ncu --version :', _ver('ncu --version | head -n 1'))
print('nsys --version:', _ver('nsys --version | head -n 1'))

import torch
print()
print('torch         :', torch.__version__)
try:
    import triton
    print('triton        :', triton.__version__)
except Exception as e:
    print('triton        : ERROR:', e)
print('cuda avail    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device name   :', torch.cuda.get_device_name(0))
    print('device cc     :', torch.cuda.get_device_capability(0))
    print('device count  :', torch.cuda.device_count())

In [ ]:
import torch

def detect_arch() -> str:
    """Return 't4', 'b200', or 'other' based on the running GPU."""
    if not torch.cuda.is_available():
        return 'cpu'
    name = torch.cuda.get_device_name(0).lower()
    cc = torch.cuda.get_device_capability(0)
    if 't4' in name or cc == (7, 5):
        return 't4'
    if 'b200' in name or cc[0] == 10:
        return 'b200'
    if 'a100' in name:
        return 'a100'
    if 'h100' in name:
        return 'h100'
    return f'cc{cc[0]}{cc[1]}'

ARCH = detect_arch()
print('detected ARCH =', ARCH)

## 2. Shape grid + global config

Grid differs by arch:

* **T4**: `N` in {256, 1024, 4096, 16384}, `V` in {32000, 128256} = 8 shapes.
* **B200**: T4 grid + `N` in {65536, 262144}, same `V` set = 12 shapes.

Also defines the `SKIP_NCU` fallback flag and the `FORCE` override used by the sweep cells for idempotency.

In [ ]:
import pathlib

# Global configuration -- edit these to trim or extend the sweep.
IMPLS = ['pytorch', 'triton', 'cuda_naive', 'cuda_online']
SEEDS = [0, 1, 2, 3, 4]  # for parity tests
PARITY_SEED_COUNT = 5

_T4_N = [256, 1024, 4096, 16384]
_B200_N = _T4_N + [65536, 262144]
_V_LIST = [32000, 128256]

# Canonical shapes = those used in the blog's hero table.
CANONICAL_SHAPES = [
    (1024, 32000),
    (4096, 128256),
    (16384, 128256),
]

if ARCH == 'b200':
    SHAPES = [(n, v) for n in _B200_N for v in _V_LIST]
else:
    SHAPES = [(n, v) for n in _T4_N for v in _V_LIST]

# Fallback + override flags. Flip SKIP_NCU to True if counters are denied on the
# cloud box and you want to still ship nsys-only data.
SKIP_NCU = False
FORCE = False       # re-run a shape even if its CSV already exists
WARMUP = 10
ITERS = 100

DATA_DIR = pathlib.Path('data') / ARCH
RAW_DIR = DATA_DIR / 'raw'
BLOG_IMG = pathlib.Path('blog') / 'img'
for p in (RAW_DIR, BLOG_IMG):
    p.mkdir(parents=True, exist_ok=True)

print(f'ARCH      : {ARCH}')
print(f'SHAPES    : {len(SHAPES)} shapes = {SHAPES}')
print(f'IMPLS     : {IMPLS}')
print(f'DATA_DIR  : {DATA_DIR}')
print(f'RAW_DIR   : {RAW_DIR}')
print(f'SKIP_NCU  : {SKIP_NCU}')

## 3. Compile + parity + smoke

Triggers the first `torch.utils.cpp_extension.load_inline` compile of both CUDA kernels, runs the Triton autotune once, then does 5 seeds x full grid `torch.allclose` vs PyTorch.

Tolerance is `atol=1e-3, rtol=1e-3`. fp16 logits with fp32 accumulation produce small but real algorithm-dependent differences between the four impls; tight tolerances would spuriously flag correct kernels.

In [ ]:
import torch
from bench.save_tensor import make_tensors
from kernels.fused_xent_pytorch import fused_xent_pytorch
from kernels.fused_xent_triton import fused_xent_triton
from kernels.cuda_launcher import fused_xent_cuda_naive, fused_xent_cuda_online, get_module

# JIT-compile the CUDA extension here so the compile cost doesn't pollute any
# subsequent timing cell. First call triggers nvcc; subsequent calls are cached.
print('compiling CUDA extension ...')
get_module(verbose=True)
print('ok')

# Tiny smoke shape.
logits, targets = make_tensors(N=64, V=1024, seed=0)
for name, fn in [
    ('pytorch',     fused_xent_pytorch),
    ('triton',      fused_xent_triton),
    ('cuda_naive',  fused_xent_cuda_naive),
    ('cuda_online', fused_xent_cuda_online),
]:
    loss = fn(logits, targets)
    torch.cuda.synchronize()
    print(f'{name:12s} loss[:5] =', loss[:5].tolist())
print('SMOKE OK')

In [ ]:
from bench.save_tensor import make_tensors
from kernels.fused_xent_pytorch import fused_xent_pytorch
from kernels.fused_xent_triton import fused_xent_triton
from kernels.cuda_launcher import fused_xent_cuda_naive, fused_xent_cuda_online

# A compact parity grid: keep small N for speed, cover both V sizes, 5 seeds.
PARITY_SHAPES = [
    (N, V) for N in (64, 256, 1024, 4096) for V in (1024, 32000, 128256)
]

ATOL, RTOL = 1e-3, 1e-3
CUSTOMS = [
    ('triton',      fused_xent_triton),
    ('cuda_naive',  fused_xent_cuda_naive),
    ('cuda_online', fused_xent_cuda_online),
]

fail_count = 0
total = 0
for seed in range(PARITY_SEED_COUNT):
    for N, V in PARITY_SHAPES:
        logits, targets = make_tensors(N=N, V=V, seed=seed)
        ref = fused_xent_pytorch(logits, targets)
        for name, fn in CUSTOMS:
            total += 1
            out = fn(logits, targets)
            ok = torch.allclose(out, ref, atol=ATOL, rtol=RTOL)
            if not ok:
                fail_count += 1
                diff = (out - ref).abs()
                print(f'FAIL {name:12s} seed={seed} N={N} V={V}  max_abs={diff.max().item():.4e}  mean_abs={diff.mean().item():.4e}')
        del logits, targets, ref
        torch.cuda.empty_cache()

print(f'\nparity: {total - fail_count}/{total} passed  (atol={ATOL}, rtol={RTOL})')
assert fail_count == 0, 'parity failures; inspect the output above'

## 4. Unlock ncu counters (cloud box only)

On Colab this is a no-op (no sudo). On a B200 instance you need root access to enable persistence mode and unlock performance counters. The cell is safe to run anywhere; failures are ignored.

If the ncu smoke capture in the next section still fails after running this, flip `SKIP_NCU = True` and continue with nsys-only data.

In [ ]:
import os, platform
if platform.system() == 'Linux':
    # Best-effort: these commands need sudo and are no-ops on Colab.
    os.system('sudo -n nvidia-smi -pm 1 2>/dev/null || true')
    os.system('echo 0 | sudo -n tee /proc/sys/kernel/perf_event_paranoid >/dev/null 2>&1 || true')
    # For Windows or systems without sudo access this just prints and moves on.
    os.system('cat /proc/sys/kernel/perf_event_paranoid 2>/dev/null || true')
else:
    print('non-Linux host; skipping permission unlock')

## 5. Profile helpers + sweeps

The helpers wrap `bench/run_single.py` in `ncu` / `nsys` via `subprocess.run`. Each is idempotent: if the expected output CSV already exists and `force=False`, the call is a no-op. This lets us re-run the notebook freely after session drops.

**Timing sweep** is the fast path (no profiler overhead) and uses the full `--iters 100` budget. Results go to `data/{arch}/timing.csv`.

**ncu sweep** uses `--iters 1 --no-warmup` because ncu replays each kernel for counter collection; multiple iterations just slow it down. One pair of (kernel, metric-set) at a time.

**nsys sweep** runs `--iters 100` to get a stable kernel-summary distribution.

In [ ]:
import json, os, shlex, subprocess, sys, pathlib
import pandas as pd
from tqdm.auto import tqdm

PY = sys.executable  # use the same interpreter as the notebook
RUN_SINGLE = 'bench/run_single.py'

def _run(cmd: str, check: bool = True, timeout: int = 600):
    """Run a shell command; raise on nonzero exit."""
    cp = subprocess.run(cmd, shell=True, check=False, timeout=timeout,
                        capture_output=True, text=True)
    if check and cp.returncode != 0:
        sys.stdout.write(cp.stdout)
        sys.stderr.write(cp.stderr)
        raise RuntimeError(f'command failed (rc={cp.returncode}): {cmd}')
    return cp


def run_timing(impl: str, N: int, V: int, arch: str,
               iters: int = ITERS, warmup: int = WARMUP,
               force: bool = False) -> dict | None:
    """Run timing-only (no profiler) for one (impl, N, V). Returns JSON summary."""
    out_json = RAW_DIR / f'timing_{impl}_{N}_{V}.json'
    if out_json.exists() and not force:
        return json.loads(out_json.read_text())
    cmd = f'{PY} {RUN_SINGLE} --impl {impl} -N {N} -V {V} --iters {iters} --warmup {warmup} -o {out_json}'
    _run(cmd, timeout=900)
    return json.loads(out_json.read_text()) if out_json.exists() else None


def run_ncu(impl: str, N: int, V: int, arch: str, force: bool = False) -> pathlib.Path | None:
    """ncu-wrap one (impl, N, V). Writes ncu_{impl}_{N}_{V}.csv + .ncu-rep."""
    out_stem = RAW_DIR / f'ncu_{impl}_{N}_{V}'
    out_csv = out_stem.with_suffix('.csv')
    if out_csv.exists() and not force:
        return out_csv
    sections = (
        '--section SchedulerStats --section WarpStateStats '
        '--section SourceCounters --section MemoryWorkloadAnalysis '
        '--section Occupancy --section ComputeWorkloadAnalysis'
    )
    inner = f'{PY} {RUN_SINGLE} --impl {impl} -N {N} -V {V} --iters 1 --no-warmup'
    # --target-processes all so we capture the subprocess Triton/torch spawns.
    cmd = (
        f'ncu --csv --log-file {out_csv} -o {out_stem} '
        f'--target-processes all {sections} '
        f'-k regex:"cross_entropy|xent|fused_xent" '
        f'-f {inner}'  # -f = overwrite existing .ncu-rep
    )
    _run(cmd, timeout=1800)
    return out_csv


def run_nsys(impl: str, N: int, V: int, arch: str, force: bool = False) -> pathlib.Path | None:
    """nsys-wrap one (impl, N, V); also emits the two per-report CSVs we merge."""
    out_stem = RAW_DIR / f'nsys_{impl}_{N}_{V}'
    kern_csv = out_stem.parent / f'{out_stem.name}_cuda_gpu_kern_sum.csv'
    if kern_csv.exists() and not force:
        return kern_csv
    inner = f'{PY} {RUN_SINGLE} --impl {impl} -N {N} -V {V} --iters {ITERS} --warmup {WARMUP}'
    _run(
        f'nsys profile --trace=cuda,nvtx,osrt '
        f'--output={out_stem} --force-overwrite=true '
        f'{inner}',
        timeout=1800,
    )
    _run(
        f'nsys stats --report cuda_gpu_kern_sum --report cuda_api_sum '
        f'--format csv --output {out_stem} {out_stem}.nsys-rep',
        timeout=600,
    )
    return kern_csv


print('helpers loaded:', [fn.__name__ for fn in (run_timing, run_ncu, run_nsys)])

In [ ]:
# Fast path: no profiler, just torch.cuda.Event timing. Target is <10 min even on T4.

rows = []
errors = 0
pbar = tqdm(total=len(IMPLS) * len(SHAPES), desc='timing sweep')
for impl in IMPLS:
    for N, V in SHAPES:
        try:
            summary = run_timing(impl, N, V, ARCH, force=FORCE)
            if summary is None:
                pbar.update(1); continue
            rows.append({
                'impl': impl, 'N': N, 'V': V,
                'mean_ms': summary['mean_ms'], 'std_ms': summary['std_ms'],
                'p50_ms': summary['p50_ms'], 'p95_ms': summary['p95_ms'], 'p99_ms': summary['p99_ms'],
                'min_ms': summary['min_ms'], 'max_ms': summary['max_ms'],
                'device': summary['device'], 'cc': summary['compute_capability'],
            })
        except Exception as e:
            errors += 1
            print(f'[timing] {impl} N={N} V={V} -> ERROR: {e}')
        pbar.update(1)
pbar.close()

if not rows:
    raise RuntimeError(
        f'timing sweep produced 0 rows ({errors} errors). '
        'Check the tracebacks above and fix before re-running. '
        'Most common cause: stale bench/run_single.py after git pull -- restart the kernel.'
    )

timing = pd.DataFrame(rows).sort_values(['impl', 'N', 'V']).reset_index(drop=True)
timing.to_csv(DATA_DIR / 'timing.csv', index=False)
print(f'wrote {DATA_DIR / "timing.csv"}: {len(timing)} rows  (errors: {errors})')
timing.head(20)

In [ ]:
# ncu sweep: idempotent, heavy. Expect ~1-3 min per (impl, shape) capture.
# Skip entirely if SKIP_NCU is set.

if SKIP_NCU:
    print('SKIP_NCU=True -> not running ncu captures')
else:
    pbar = tqdm(total=len(IMPLS) * len(SHAPES), desc='ncu sweep')
    for impl in IMPLS:
        for N, V in SHAPES:
            try:
                run_ncu(impl, N, V, ARCH, force=FORCE)
            except Exception as e:
                print(f'[ncu] {impl} N={N} V={V} -> ERROR: {e}')
            pbar.update(1)
    pbar.close()
    print('ncu sweep done')

## 6. nsys sweep (and optional CUDA Graphs variant)

nsys captures kernel-level timing + CUDA API overhead at ~1% runtime overhead, so we run it with the full iteration count.

In [ ]:
pbar = tqdm(total=len(IMPLS) * len(SHAPES), desc='nsys sweep')
for impl in IMPLS:
    for N, V in SHAPES:
        try:
            run_nsys(impl, N, V, ARCH, force=FORCE)
        except Exception as e:
            print(f'[nsys] {impl} N={N} V={V} -> ERROR: {e}')
        pbar.update(1)
pbar.close()
print('nsys sweep done')

In [ ]:
# OPTIONAL (B200 bonus): CUDA Graph-captured variant of each impl.
# Measures launch-overhead amortization at high kernel-launch rates.
# Writes data/{arch}/timing_graphs.csv.

import time
from bench.save_tensor import make_tensors
from kernels.fused_xent_pytorch import fused_xent_pytorch
from kernels.fused_xent_triton import fused_xent_triton
from kernels.cuda_launcher import fused_xent_cuda_naive, fused_xent_cuda_online

FNS = {
    'pytorch':     fused_xent_pytorch,
    'triton':      fused_xent_triton,
    'cuda_naive':  fused_xent_cuda_naive,
    'cuda_online': fused_xent_cuda_online,
}

def time_with_graph(fn, logits, targets, iters=100, warmup=10):
    # Warmup in a side stream before capture (required by CUDA Graphs docs).
    s = torch.cuda.Stream()
    s.wait_stream(torch.cuda.current_stream())
    with torch.cuda.stream(s):
        for _ in range(warmup):
            _ = fn(logits, targets)
    torch.cuda.current_stream().wait_stream(s)

    g = torch.cuda.CUDAGraph()
    with torch.cuda.graph(g):
        _ = fn(logits, targets)

    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        g.replay()
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    return (t1 - t0) / iters * 1e3  # ms/iter

rows = []
for impl in IMPLS:
    for N, V in CANONICAL_SHAPES:
        try:
            logits, targets = make_tensors(N=N, V=V, seed=0)
            # Non-graphed baseline from the timing sweep is already in timing.csv.
            ms_graph = time_with_graph(FNS[impl], logits, targets)
            rows.append({'impl': impl, 'N': N, 'V': V, 'graph_ms': ms_graph})
            del logits, targets
            torch.cuda.empty_cache()
        except Exception as e:
            print(f'[graphs] {impl} N={N} V={V} -> ERROR: {e}')

graphs = pd.DataFrame(rows)
graphs.to_csv(DATA_DIR / 'timing_graphs.csv', index=False)
graphs

## 7. Bonus experiments (run exactly one on B200)

* **7a Autotune amortization**: Triton's autotune picks the best config on the first call of each shape. The warm cost is zero; the cold cost can be seconds. We measure how many launches are needed to amortize the first-call autotune cost vs a hand-tuned CUDA kernel.
* **7b NVML power sampling**: sample `nvmlDeviceGetPowerUsage` at ~100 Hz during a sustained run; compute pJ/row.

Only run one; skip the other. Outputs land in `data/{arch}/bonus_*.csv`.

In [ ]:
# 7a: Autotune amortization. Uses large shape where kernel time dominates.

import importlib, time
import torch
from bench.save_tensor import make_tensors

# Pick a biggish shape that fits on this GPU.
N_auto, V_auto = (16384, 128256) if ARCH == 't4' else (65536, 128256)
logits, targets = make_tensors(N=N_auto, V=V_auto, seed=0)

# Cold Triton (reload module to nuke autotune cache in this process).
if 'kernels.fused_xent_triton' in sys.modules:
    del sys.modules['kernels.fused_xent_triton']
from kernels.fused_xent_triton import fused_xent_triton as _triton

rows = []
# Cold first call.
torch.cuda.synchronize(); t0 = time.perf_counter()
_ = _triton(logits, targets)
torch.cuda.synchronize(); cold_ms = (time.perf_counter() - t0) * 1e3
rows.append({'impl': 'triton', 'call': 1, 'phase': 'cold', 'ms': cold_ms})

# Warm subsequent calls.
for i in range(2, 11):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    _ = _triton(logits, targets)
    torch.cuda.synchronize(); ms = (time.perf_counter() - t0) * 1e3
    rows.append({'impl': 'triton', 'call': i, 'phase': 'warm', 'ms': ms})

# CUDA baseline for reference (first-call JIT cost was already paid above).
from kernels.cuda_launcher import fused_xent_cuda_online as _cuda
for i in range(1, 11):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    _ = _cuda(logits, targets)
    torch.cuda.synchronize(); ms = (time.perf_counter() - t0) * 1e3
    rows.append({'impl': 'cuda_online', 'call': i, 'phase': 'warm', 'ms': ms})

autotune = pd.DataFrame(rows)
autotune.to_csv(DATA_DIR / 'bonus_autotune.csv', index=False)
del logits, targets; torch.cuda.empty_cache()
autotune

In [ ]:
# 7b: NVML power sampling. Requires pynvml (pip install pynvml).
# Samples GPU power at ~100 Hz while a sustained loop runs.

try:
    import pynvml
except Exception:
    print('pynvml not installed; `!pip install pynvml` and re-run if you want this bonus')
    raise SystemExit

import threading, time, statistics
import torch
from bench.save_tensor import make_tensors
from kernels.fused_xent_pytorch import fused_xent_pytorch
from kernels.fused_xent_triton import fused_xent_triton
from kernels.cuda_launcher import fused_xent_cuda_naive, fused_xent_cuda_online

FNS = {
    'pytorch':     fused_xent_pytorch,
    'triton':      fused_xent_triton,
    'cuda_naive':  fused_xent_cuda_naive,
    'cuda_online': fused_xent_cuda_online,
}

pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)

def sample_power(samples, stop_flag, interval=0.01):
    while not stop_flag['stop']:
        samples.append(pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0)  # W
        time.sleep(interval)

Np, Vp = (16384, 128256) if ARCH == 't4' else (65536, 128256)
logits, targets = make_tensors(N=Np, V=Vp, seed=0)

rows = []
for impl, fn in FNS.items():
    for _ in range(10):
        fn(logits, targets)
    torch.cuda.synchronize()
    samples, stop = [], {'stop': False}
    t = threading.Thread(target=sample_power, args=(samples, stop))
    t.start()
    t0 = time.perf_counter()
    iters = 200
    for _ in range(iters):
        fn(logits, targets)
    torch.cuda.synchronize()
    wall = time.perf_counter() - t0
    stop['stop'] = True
    t.join()
    avg_w = statistics.mean(samples) if samples else 0.0
    energy_j = avg_w * wall
    pj_per_row = (energy_j / (iters * Np)) * 1e12
    rows.append({'impl': impl, 'N': Np, 'V': Vp,
                 'iters': iters, 'wall_s': wall, 'avg_W': avg_w,
                 'energy_J': energy_j, 'pJ_per_row': pj_per_row,
                 'samples': len(samples)})

power = pd.DataFrame(rows)
power.to_csv(DATA_DIR / 'bonus_power.csv', index=False)
pynvml.nvmlShutdown()
del logits, targets; torch.cuda.empty_cache()
power

## 8. Merge raw CSVs into tidy tables

Consumes `data/{arch}/raw/{ncu,nsys}_*.csv`. Writes `ncu_merged.csv`, `nsys_merged.csv`, `nsys_api_merged.csv` to `data/{arch}/`.

In [ ]:
import importlib
from bench import merge_csvs
importlib.reload(merge_csvs)

merged = merge_csvs.merge_all(DATA_DIR)
for name, df in merged.items():
    print(f'{name}: {len(df)} rows, {len(df.columns)} cols')
ncu, nsys_kern, nsys_api = merged['ncu'], merged['nsys_kern'], merged['nsys_api']
display(ncu.head() if not ncu.empty else 'no ncu data')
display(nsys_kern.head() if not nsys_kern.empty else 'no nsys data')

## 9. Plots

All figures saved to `blog/img/`. The first four are per-arch; the fifth (cross-arch heatmap) only fires when both `data/t4/` and `data/b200/` exist.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 150,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

IMPL_ORDER = ['pytorch', 'triton', 'cuda_naive', 'cuda_online']
IMPL_COLORS = {
    'pytorch':     '#7f7f7f',
    'triton':      '#1f77b4',
    'cuda_naive':  '#d62728',
    'cuda_online': '#2ca02c',
}

def _save(fig, name):
    path = BLOG_IMG / f'{ARCH}_{name}.png'
    fig.savefig(path, bbox_inches='tight')
    print('wrote', path)

timing = pd.read_csv(DATA_DIR / 'timing.csv') if (DATA_DIR / 'timing.csv').exists() else pd.DataFrame()
timing.head() if not timing.empty else 'run section 5 first'

In [ ]:
# Plot 1: per-shape mean latency bar chart, one subplot per vocab size.
if timing.empty:
    print('timing.csv missing; skipping'); 
else:
    Vs = sorted(timing['V'].unique())
    fig, axes = plt.subplots(1, len(Vs), figsize=(6*len(Vs), 4.5), sharey=False)
    if len(Vs) == 1:
        axes = [axes]
    for ax, V in zip(axes, Vs):
        sub = timing[timing['V'] == V].copy()
        Ns = sorted(sub['N'].unique())
        width = 0.8 / len(IMPL_ORDER)
        for i, impl in enumerate(IMPL_ORDER):
            vals = [sub[(sub['N']==N) & (sub['impl']==impl)]['mean_ms'].mean() for N in Ns]
            x = np.arange(len(Ns)) + (i - (len(IMPL_ORDER)-1)/2) * width
            ax.bar(x, vals, width=width, label=impl, color=IMPL_COLORS[impl])
        ax.set_xticks(np.arange(len(Ns)))
        ax.set_xticklabels([f'N={n}' for n in Ns])
        ax.set_ylabel('mean ms / call')
        ax.set_title(f'V = {V}')
        ax.set_yscale('log')
        ax.legend()
    fig.suptitle(f'Fused cross-entropy latency ({ARCH.upper()})')
    plt.tight_layout()
    _save(fig, 'latency_bar')
    plt.show()

In [ ]:
# Plot 2: tail-latency (p50 / p95 / p99) per impl at the largest shape.
if not timing.empty:
    V_big = int(timing['V'].max())
    N_big = int(timing[timing['V']==V_big]['N'].max())
    sub = timing[(timing['N']==N_big) & (timing['V']==V_big)].copy()
    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(len(IMPL_ORDER))
    width = 0.25
    for j, p in enumerate(['p50_ms', 'p95_ms', 'p99_ms']):
        vals = [sub[sub['impl']==impl][p].mean() for impl in IMPL_ORDER]
        ax.bar(x + (j-1)*width, vals, width=width, label=p)
    ax.set_xticks(x); ax.set_xticklabels(IMPL_ORDER, rotation=20)
    ax.set_ylabel('ms / call')
    ax.set_title(f'Tail latency at N={N_big}, V={V_big} ({ARCH.upper()})')
    ax.legend()
    plt.tight_layout()
    _save(fig, 'tail_latency')
    plt.show()

In [ ]:
# Plot 3: DRAM bytes read + write per impl (aggregated across all shapes).
if not ncu.empty and 'dram__bytes_read.sum' in ncu.columns:
    agg = ncu.groupby('impl')[['dram__bytes_read.sum', 'dram__bytes_write.sum']].sum().reindex(IMPL_ORDER)
    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(len(agg))
    ax.bar(x - 0.2, agg['dram__bytes_read.sum']/1e9, width=0.4, label='read (GB)')
    ax.bar(x + 0.2, agg['dram__bytes_write.sum']/1e9, width=0.4, label='write (GB)')
    ax.set_xticks(x); ax.set_xticklabels(agg.index, rotation=20)
    ax.set_ylabel('DRAM bytes (GB)')
    ax.set_title(f'DRAM bytes per impl ({ARCH.upper()}) - lower is better')
    ax.legend()
    plt.tight_layout()
    _save(fig, 'dram_bytes')
    plt.show()
else:
    print('ncu DRAM metrics missing; run section 5 with SKIP_NCU=False')

In [ ]:
# Plot 4: warp-stall-reason stacked bar per impl.
stall_cols = [
    ('smsp__warps_issue_stalled_long_scoreboard_per_warp_active.pct', 'long scoreboard'),
    ('smsp__warps_issue_stalled_mio_throttle_per_warp_active.pct',    'mio throttle'),
    ('smsp__warps_issue_stalled_math_throttle_per_warp_active.pct',   'math throttle'),
    ('smsp__warps_issue_stalled_wait_per_warp_active.pct',            'wait'),
    ('smsp__warps_issue_stalled_barrier_per_warp_active.pct',         'barrier'),
]
have = [c for c, _ in stall_cols if c in ncu.columns]
if have and not ncu.empty:
    sub = ncu.groupby('impl')[[c for c,_ in stall_cols if c in ncu.columns]].mean().reindex(IMPL_ORDER)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    bottom = np.zeros(len(sub))
    for col, label in stall_cols:
        if col not in sub.columns: continue
        vals = sub[col].fillna(0).values
        ax.bar(sub.index, vals, bottom=bottom, label=label)
        bottom += vals
    ax.set_ylabel('warp stall % per active warp')
    ax.set_title(f'Warp stall reasons ({ARCH.upper()}) - what each impl waits on')
    ax.legend(loc='upper right', fontsize=9)
    plt.xticks(rotation=20)
    plt.tight_layout()
    _save(fig, 'stall_reasons')
    plt.show()
else:
    print('ncu stall metrics missing; run section 5 with SKIP_NCU=False')

In [ ]:
# Hero table for the blog's headline: 4 canonical shapes x 4 impls, mean ms and speedup vs PyTorch.
if not timing.empty:
    hero = []
    for N, V in CANONICAL_SHAPES:
        sub = timing[(timing['N']==N) & (timing['V']==V)]
        pyt = sub[sub['impl']=='pytorch']['mean_ms'].mean()
        row = {'N': N, 'V': V}
        for impl in IMPL_ORDER:
            ms = sub[sub['impl']==impl]['mean_ms'].mean()
            row[f'{impl}_ms'] = ms
            row[f'{impl}_speedup'] = (pyt / ms) if (pyt and ms) else float('nan')
        hero.append(row)
    hero_df = pd.DataFrame(hero)
    hero_df.to_csv(DATA_DIR / 'hero_table.csv', index=False)
    display(hero_df)

## 10. Cross-architecture plots (T4 + B200 side-by-side)

Only runs if both `data/t4/timing.csv` and `data/b200/timing.csv` exist. Producing the cross-arch heatmap is the whole point of the study -- it's the hero figure for the blog.

In [ ]:
t4_path   = pathlib.Path('data/t4/timing.csv')
b200_path = pathlib.Path('data/b200/timing.csv')
if not (t4_path.exists() and b200_path.exists()):
    print('cross-arch plot skipped: need both', t4_path, 'and', b200_path)
else:
    t4   = pd.read_csv(t4_path).assign(arch='T4')
    b200 = pd.read_csv(b200_path).assign(arch='B200')
    both = pd.concat([t4, b200], ignore_index=True)

    # Per (arch, shape), identify the winning impl.
    winners = (both.sort_values('mean_ms')
                   .groupby(['arch', 'N', 'V'], as_index=False)
                   .first()[['arch','N','V','impl','mean_ms']])

    # Heatmap-like categorical plot.
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, arch in zip(axes, ['T4', 'B200']):
        sub = winners[winners['arch']==arch]
        if sub.empty:
            ax.set_title(f'{arch} (no data)'); continue
        pivot = sub.pivot(index='N', columns='V', values='impl')
        # Map impl names to integers for imshow.
        labels = IMPL_ORDER
        code = {l:i for i,l in enumerate(labels)}
        Z = pivot.applymap(lambda x: code.get(x, -1)).values
        im = ax.imshow(Z, aspect='auto', cmap='viridis')
        ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
        ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
        ax.set_xlabel('V'); ax.set_ylabel('N')
        ax.set_title(f'Winning impl per shape ({arch})')
        for i in range(Z.shape[0]):
            for j in range(Z.shape[1]):
                if Z[i,j] >= 0:
                    ax.text(j, i, labels[int(Z[i,j])], ha='center', va='center', fontsize=9, color='white')
    plt.tight_layout()
    fig.savefig('blog/img/crossarch_winner_heatmap.png', bbox_inches='tight', dpi=150)
    print('wrote blog/img/crossarch_winner_heatmap.png')
    plt.show()

## 11. Export data + (optional) git push

Convenient on a cloud VM: tarball `data/{arch}/` so we can `rsync` the whole thing back with one command, and optionally `git push` the merged CSVs + figures from inside the box as a backup path before terminating.

In [ ]:
import subprocess, datetime, pathlib
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
out = f'data_{ARCH}_{ts}.tgz'
subprocess.run(f'tar czf {out} data/{ARCH} blog/img', shell=True, check=True)
print('wrote', out, f'({pathlib.Path(out).stat().st_size / 1e6:.1f} MB)')

In [ ]:
# OPTIONAL: commit the merged CSVs + figures back to the repo.
# Only works if this VM has git credentials for the remote.

DO_PUSH = False  # flip to True on the B200 box
if DO_PUSH:
    import subprocess
    subprocess.run('git add data/*/ncu_merged.csv data/*/nsys_merged.csv data/*/nsys_api_merged.csv data/*/timing.csv data/*/timing_graphs.csv data/*/bonus_*.csv data/*/hero_table.csv blog/img/*.png',
                   shell=True, check=False)
    subprocess.run(f'git commit -m "bench({ARCH}): sweep results"', shell=True, check=False)
    subprocess.run('git push', shell=True, check=False)